In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\swatisingh\\ml\\mlops_project\\kidney-disease-classification-project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int
    


In [4]:
from kidney_disease_classifier.constants import *
from kidney_disease_classifier.utils.common import read_yaml, create_directories

In [5]:
class configuration_manager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        params = self.params
        
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate=params.LEARNING_RATE,
            params_include_top=params.INCLUDE_TOP,
            params_weights=params.WEIGHTS,
            params_classes=params.CLASSES
        )
        
        return prepare_base_model_config

In [6]:
import torch
from torchvision import models
from torch import nn
from pathlib import Path

class PrepareBaseModel:
    def __init__(self, config):
        self.config = config

    def get_base_model(self):
        """
        Load pretrained VGG16
        """
        self.model = models.vgg16(
            weights=models.VGG16_Weights.IMAGENET1K_V1
        )

        self.save_model(self.model, self.config.base_model_path)
        return self.model

    def update_base_model(self):
        """
        Freeze layers + modify classifier
        """

        # Freeze all layers
        for param in self.model.parameters():
            param.requires_grad = False

        # Replace final layer
        num_features = self.model.classifier[6].in_features

        self.model.classifier[6] = nn.Linear(
            in_features=num_features,
            out_features=self.config.params_classes
        )

        self.save_model(self.model, self.config.updated_base_model_path)

        return self.model

    @staticmethod
    def save_model(model: nn.Module, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), path)

In [8]:
try:
    config_manager = configuration_manager()
    prepare_base_model_config = config_manager.get_prepare_base_model_config()

    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)

    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()

except Exception as e:
    raise e

[2026-03-02 01:01:54,225: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-02 01:01:54,246: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-02 01:01:54,248: INFO: common: Directory created at: artifacts]
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [00:45<00:00, 12.1MB/s] 
